# M5 thesis — V2 hyperparameter tuning (corrected)
**What changed vs July:** each product's history starts at its first sale (M5 RMSSE scale, SBC classes, training rows); only the 1,913-day file; tuning windows end on day 1,745 (before every test day);
each product gets its own model (as in the evaluation); the score is the mean RMSSE over all tuning products of the class;
classes computed on days 1–1,745 only; seeded search; the full trial history is saved (convergence curves).
Run the cells in order. The tuning resumes automatically after a disconnection.

In [1]:
# Cell 1 — Setup
from google.colab import drive
drive.mount('/content/drive')
!pip install catboost optuna -q
import os, sys, json, time, glob, platform
D = '/content/drive/MyDrive/M5_Thesis'
V2 = f'{D}/results/v2'; os.makedirs(f'{V2}/tuning', exist_ok=True)


Mounted at /content/drive


In [2]:
# Cell 2 — Pipeline V2 definitions (same code in every V2 notebook)
# ============================================================================
# PIPELINE V2 — shared definitions (identical in the tuning and evaluation notebooks)
# ============================================================================
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Lasso, Ridge, ElasticNet
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

N_DAYS     = 1913                       # sales_train_validation.csv only
MAX_LAG    = 42
FIRST_ROW_DAY = MAX_LAG + 1             # row r forecasts day r + 43 (1-based)
H          = 28                         # window length (days)
CUT        = 1745                       # last day used for classes and characteristics
TUNE_STARTS = [1662, 1690, 1718]        # tuning windows: days 1,662-1,745 (before any test day)
EVAL_STARTS = [1746, 1774, 1802, 1830, 1858]   # 5 test windows: days 1,746-1,885
HOLD_START  = 1886                      # holdout window: days 1,886-1,913
MODELS = ['LGBM', 'XGBOOST', 'CATBOOST', 'HGBR', 'RANDOM_FOREST',
          'ELASTIC_NET', 'EXTRA_TREES', 'GRADIENT_BOOST', 'RIDGE', 'LASSO']
LINEAR = {'RIDGE', 'LASSO', 'ELASTIC_NET'}
N_TRIALS = {'RIDGE': 30, 'LASSO': 30, 'ELASTIC_NET': 30, 'RANDOM_FOREST': 30, 'EXTRA_TREES': 30,
            'LGBM': 50, 'XGBOOST': 50, 'CATBOOST': 50, 'HGBR': 50, 'GRADIENT_BOOST': 50}

def build_features(series):
    """Row for target day s uses only days before s. Lags 7..42 fall on the SAME weekday as s.
    Rolling mean/std (population std, ddof=0) over the w days before s. Vectorised with cumulative sums."""
    x = np.asarray(series, float)
    s_idx = np.arange(MAX_LAG, len(x))
    c1 = np.r_[0.0, np.cumsum(x)]; c2 = np.r_[0.0, np.cumsum(x * x)]
    cols = [x[s_idx - k] for k in (7, 14, 28, 35, 42)]
    def mean_w(w): return (c1[s_idx] - c1[s_idx - w]) / w
    def std_w(w):
        m = mean_w(w); v = (c2[s_idx] - c2[s_idx - w]) / w - m * m
        return np.sqrt(np.clip(v, 0, None))
    cols += [mean_w(w) for w in (7, 14, 28)] + [std_w(w) for w in (7, 28)]
    return np.column_stack(cols), x[s_idx]

def first_sale_row(y):
    """First row whose target is a sale (the product's history starts at its first sale, as in the M5 RMSSE)."""
    nz = np.flatnonzero(y > 0)
    return int(nz[0]) if len(nz) else len(y)

def window_rows(start_day):
    """Training rows = all target days before start_day; test rows = the 28 days from start_day."""
    a = start_day - FIRST_ROW_DAY
    return a, a + H                      # train = rows[:a], test = rows[a:a+H]

def rmsse(y_test, pred, y_train):
    """RMSSE with the M5 scale: mean squared one-step change of the training targets from the first sale onward."""
    t = y_train[first_sale_row(y_train):]
    if len(t) < 2: return np.nan
    scale = np.mean(np.diff(t) ** 2)
    return np.nan if scale == 0 else float(np.sqrt(np.mean((y_test - pred) ** 2) / scale))

def window_ok(y, start_day, min_rows=50):
    """A window is evaluable if the product has >= min_rows training rows after its first sale and a non-zero scale."""
    a, _ = window_rows(start_day)
    t = y[first_sale_row(y[:a]):a]
    return len(t) >= min_rows and np.any(np.diff(t) != 0)

def build_model(name, p, n_jobs=-1):
    p = dict(p)
    if name == 'LGBM':          return lgb.LGBMRegressor(objective='regression', verbosity=-1, n_jobs=n_jobs, random_state=42, n_estimators=200, subsample_freq=1, **p)
    if name == 'XGBOOST':       return XGBRegressor(objective='reg:squarederror', verbosity=0, n_jobs=n_jobs, random_state=42, n_estimators=200, **p)
    if name == 'CATBOOST':      return CatBoostRegressor(verbose=False, random_seed=42, iterations=200, thread_count=n_jobs, allow_writing_files=False, **p)
    if name == 'HGBR':          return HistGradientBoostingRegressor(random_state=42, max_iter=200, **p)
    if name == 'RANDOM_FOREST': return RandomForestRegressor(random_state=42, n_jobs=n_jobs, **p)
    if name == 'EXTRA_TREES':   return ExtraTreesRegressor(random_state=42, n_jobs=n_jobs, **p)
    if name == 'GRADIENT_BOOST':return GradientBoostingRegressor(random_state=42, n_estimators=200, **p)
    if name == 'ELASTIC_NET':   return ElasticNet(max_iter=10000, random_state=42, **p)
    if name == 'LASSO':         return Lasso(max_iter=10000, random_state=42, **p)
    if name == 'RIDGE':         return Ridge(**p)

def fit_predict(name, p, X, y, start_day, n_jobs=-1):
    """Train on the product's rows from its first sale up to the day before start_day; forecast the 28-day window."""
    a, b = window_rows(start_day)
    f = first_sale_row(y[:a])
    Xtr, ytr, Xte, yte = X[f:a], y[f:a], X[a:b], y[a:b]
    if name in LINEAR:
        sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
    m = build_model(name, p, n_jobs); m.fit(Xtr, ytr)
    pred = np.clip(m.predict(Xte), 0, None)
    return yte, pred, ytr

def search_space(name, trial):          # identical ranges to the July tuning (cells 85-105)
    s = trial.suggest_float; i = trial.suggest_int
    if name == 'LASSO':  return {'alpha': s('alpha', 1e-4, 100.0, log=True)}
    if name == 'RIDGE':  return {'alpha': s('alpha', 1e-4, 1000.0, log=True)}
    if name == 'ELASTIC_NET': return {'alpha': s('alpha', 1e-4, 100.0, log=True), 'l1_ratio': s('l1_ratio', 0.0, 1.0)}
    if name == 'LGBM':   return {'learning_rate': s('learning_rate', 0.01, 0.30), 'num_leaves': i('num_leaves', 8, 256),
                                 'min_child_samples': i('min_child_samples', 5, 200), 'reg_lambda': s('reg_lambda', 1e-3, 10.0, log=True),
                                 'subsample': s('subsample', 0.5, 1.0), 'colsample_bytree': s('colsample_bytree', 0.5, 1.0)}
    if name == 'XGBOOST':return {'learning_rate': s('learning_rate', 0.01, 0.30), 'max_depth': i('max_depth', 3, 10),
                                 'min_child_weight': i('min_child_weight', 5, 100), 'reg_lambda': s('reg_lambda', 1e-3, 10.0, log=True),
                                 'subsample': s('subsample', 0.5, 1.0), 'colsample_bytree': s('colsample_bytree', 0.5, 1.0)}
    if name == 'CATBOOST': return {'learning_rate': s('learning_rate', 0.01, 0.30), 'depth': i('depth', 3, 10),
                                   'l2_leaf_reg': s('l2_leaf_reg', 1e-3, 10.0, log=True)}
    if name == 'HGBR':   return {'learning_rate': s('learning_rate', 0.01, 0.30), 'max_depth': i('max_depth', 2, 15),
                                 'min_samples_leaf': i('min_samples_leaf', 5, 200), 'l2_regularization': s('l2_regularization', 1e-3, 10.0, log=True)}
    if name == 'GRADIENT_BOOST': return {'learning_rate': s('learning_rate', 0.01, 0.30), 'max_depth': i('max_depth', 2, 8),
                                         'subsample': s('subsample', 0.5, 1.0), 'min_samples_leaf': i('min_samples_leaf', 5, 100)}
    if name in ('RANDOM_FOREST', 'EXTRA_TREES'):
        return {'n_estimators': i('n_estimators', 50, 300), 'max_depth': i('max_depth', 3, 30),
                'max_features': s('max_features', 0.3, 1.0), 'min_samples_leaf': i('min_samples_leaf', 5, 100)}

def sbc_class(series_until_cut):
    """SBC (2005) on the product's history from its first sale: p = T / N (T = days since first sale, N = days with sales),
    CV2 of non-zero demand (sample variance, ddof=1); cut-offs 1.32 and 0.49. One sale only -> CV2 = 0. Fewer than 50 days since the first sale -> SHORT_HISTORY."""
    y = np.asarray(series_until_cut, float); nz = np.flatnonzero(y > 0)
    if len(nz) == 0: return 'NO_SALE', np.nan, np.nan
    y = y[nz[0]:]; pos = y[y > 0]
    p = len(y) / len(pos)
    if len(y) < 50: return 'SHORT_HISTORY', p, np.nan       # fewer than 50 days since the first sale
    cv2 = pos.var(ddof=1) / pos.mean() ** 2 if len(pos) > 1 else 0.0
    cls = ('SMOOTH' if cv2 <= 0.49 else 'ERRATIC') if p <= 1.32 else ('INTERMITTENT' if cv2 <= 0.49 else 'LUMPY')
    return cls, p, cv2


In [3]:
# Cell 3 — Data, classes on days 1-1,745, library versions
import sklearn, lightgbm, xgboost, catboost, optuna, scipy
sales = pd.read_csv(f'{D}/data/sales_train_validation.csv').set_index('id')
day_cols = [c for c in sales.columns if c.startswith('d_')]
assert len(day_cols) == N_DAYS, 'Wrong file: expected 1,913 days'
Y = sales[day_cols].to_numpy(float)
print(f'✅ Sales: {Y.shape[0]:,} products × {Y.shape[1]:,} days')
rows = [(sku, *sbc_class(y[:CUT])) for sku, y in zip(sales.index, Y)]
cls = pd.DataFrame(rows, columns=['SKU_ID', 'class_cut1745', 'p_cut1745', 'cv2_cut1745']).set_index('SKU_ID')
off = pd.read_csv(f'{D}/results/demand_classification_results.csv').set_index('id')['demand_category']
cls['class_official_1913'] = off.reindex(cls.index).astype(str).str.upper()
cls.to_csv(f'{V2}/classes_cut1745.csv')
print(cls['class_cut1745'].value_counts().to_string())
print(f"Agreement with the official 1,913-day classes: {(cls['class_cut1745'] == cls['class_official_1913']).mean():.2%}")
versions = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'scikit-learn': sklearn.__version__,
            'lightgbm': lightgbm.__version__, 'xgboost': xgboost.__version__, 'catboost': catboost.__version__,
            'optuna': optuna.__version__, 'scipy': scipy.__version__}
json.dump(versions, open(f'{V2}/library_versions_tuning.json', 'w'), indent=1); print(versions)


✅ Sales: 30,490 products × 1,913 days
class_cut1745
INTERMITTENT     22087
LUMPY             5509
SMOOTH            1928
ERRATIC            877
SHORT_HISTORY       53
NO_SALE             36
Agreement with the official 1,913-day classes: 92.82%
{'python': '3.13.18', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scikit-learn': '1.6.1', 'lightgbm': '4.6.0', 'xgboost': '3.4.1', 'catboost': '1.2.10', 'optuna': '5.0.0', 'scipy': '1.16.3'}


In [4]:
# Cell 4 — Tuning products (the same 390 products as in July), grouped by their class on days 1-1,745
sub = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{D}/subsamples/subsample_*.csv'))])
tune_ids = [s for s in sub['id'].unique() if s in cls.index]
tune = cls.loc[tune_ids, ['class_cut1745']]
tune = tune[~tune['class_cut1745'].isin(['NO_SALE', 'SHORT_HISTORY'])]
print(f'Tuning products: {len(tune)}'); print(tune['class_cut1745'].value_counts().to_string())
FEAT = {s: build_features(Y[sales.index.get_loc(s)]) for s in tune.index}
print(f'Features built for {len(FEAT)} products')
tune.to_csv(f'{V2}/tuning_products.csv')


Tuning products: 389
class_cut1745
INTERMITTENT    146
LUMPY            98
SMOOTH           87
ERRATIC          58
Features built for 389 products


In [ ]:
# Cell 5 — Tuning: 10 models × 4 classes (resumes automatically; about 1-2 days in total)
from joblib import Parallel, delayed
optuna.logging.set_verbosity(optuna.logging.WARNING)
def product_score(name, params, X, y, windows):
    vals = []
    for d0 in windows:                                   # only evaluable windows (fixed in advance)
        yte, pred, ytr = fit_predict(name, params, X, y, d0, n_jobs=1)
        vals.append(rmsse(yte, pred, ytr))
    return np.mean(vals)                                 # NaN if a prediction failed -> trial penalised below
VALID_W = {s: [d0 for d0 in TUNE_STARTS if window_ok(FEAT[s][1], d0)] for s in FEAT}
def done_ok(path):
    try: json.load(open(path)); return True
    except Exception: return False
CLASSES = ['INTERMITTENT', 'LUMPY', 'SMOOTH', 'ERRATIC']
for c in CLASSES:
    ids = [s for s in tune.index[tune['class_cut1745'] == c] if VALID_W[s]]
    print(f'{c}: {len(ids)} tuning products with at least one evaluable window')
    for name in MODELS:
        out = f'{V2}/tuning/{c}_{name}.json'
        if os.path.exists(out) and done_ok(out):
            print(f'{c:<13} {name:<15} already done'); continue
        t0 = time.time()
        def objective(trial):
            p = search_space(name, trial)
            sc = np.array(Parallel(n_jobs=-1)(delayed(product_score)(name, p, *FEAT[s], VALID_W[s]) for s in ids), float)
            n_ok = int(np.isfinite(sc).sum()); trial.set_user_attr('n_valid', n_ok); trial.set_user_attr('scores', [round(float(v), 5) for v in sc])
            return float(np.mean(sc)) if n_ok == len(ids) else float('inf')   # same products in every trial
        study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(objective, n_trials=N_TRIALS[name], catch=(Exception,))
        hist = study.trials_dataframe(); hist.to_csv(f'{V2}/tuning/{c}_{name}_trials.csv', index=False)
        n_complete = sum(t.state.name == 'COMPLETE' and np.isfinite(t.value) for t in study.trials)
        if n_complete == 0:
            raise RuntimeError(f'{c} {name}: no successful trial — check the trials CSV')
        res = {'class': c, 'model': name, 'best_params': study.best_params, 'best_mean_rmsse': round(study.best_value, 5),
               'n_trials': len(study.trials), 'n_complete': int(n_complete), 'n_products': len(ids),
               'minutes': round((time.time() - t0) / 60, 1)}
        tmp = out + '.tmp'
        with open(tmp, 'w') as fh: json.dump(res, fh, indent=1)
        os.replace(tmp, out)
        print(f"{c:<13} {name:<15} best = {res['best_mean_rmsse']:.4f}  ({res['minutes']} min, {len(ids)} products, {n_complete} complete trials)")
print('🎉 TUNING FINISHED')


INTERMITTENT: 146 tuning products with at least one evaluable window
INTERMITTENT  LGBM            already done
INTERMITTENT  XGBOOST         already done
INTERMITTENT  CATBOOST        already done
INTERMITTENT  HGBR            already done
INTERMITTENT  RANDOM_FOREST   already done
INTERMITTENT  ELASTIC_NET     already done
INTERMITTENT  EXTRA_TREES     already done
INTERMITTENT  GRADIENT_BOOST  already done
INTERMITTENT  RIDGE           already done
INTERMITTENT  LASSO           already done
LUMPY: 98 tuning products with at least one evaluable window
LUMPY         LGBM            already done
LUMPY         XGBOOST         already done
LUMPY         CATBOOST        best = 0.6125  (65.6 min, 98 products, 50 complete trials)
LUMPY         HGBR            best = 0.6131  (26.9 min, 98 products, 50 complete trials)
LUMPY         RANDOM_FOREST   best = 0.6076  (36.0 min, 98 products, 30 complete trials)
LUMPY         ELASTIC_NET     best = 0.6105  (0.5 min, 98 products, 30 complete trials)

In [ ]:
# Cell 6 — Summary and convergence (is 30 / 50 trials enough?)
rows, conv = [], []
for f in sorted(glob.glob(f'{V2}/tuning/*_trials.csv')):
    c, name = os.path.basename(f).replace('_trials.csv', '').split('_', 1)
    h = pd.read_csv(f).sort_values('number'); best = h['value'].replace([np.inf], np.nan).fillna(np.inf).cummin().replace(np.inf, np.nan)
    n = len(h); final = best.iloc[-1]
    conv.append({'class': c, 'model': name, 'trials': n, 'best after 10': best.iloc[min(9, n - 1)],
                 'best after 20': best.iloc[min(19, n - 1)], 'best after 30': best.iloc[min(29, n - 1)],
                 'final best': final, 'gain of the last third (%)': round(100 * (best.iloc[int(n * 2 / 3) - 1] - final) / final, 3)})
for f in sorted(glob.glob(f'{V2}/tuning/*.json')):
    r = json.load(open(f)); rows.append({k: r[k] for k in ['class', 'model', 'best_mean_rmsse', 'n_trials', 'n_complete', 'n_products', 'minutes']} | {'best_params': json.dumps(r['best_params'])})
summary = pd.DataFrame(rows); conv = pd.DataFrame(conv).round(5)
pd.set_option('display.width', 250); print(summary.drop(columns='best_params').to_string(index=False)); print(conv.to_string(index=False))
params_v2 = {}
for r in rows: params_v2.setdefault(r['class'], {})[r['model']] = json.loads(r['best_params'])
json.dump(params_v2, open(f'{V2}/optimal_hyperparameters_v2.json', 'w'), indent=1)
with pd.ExcelWriter(f'{V2}/Tuning_v2.xlsx') as xw:
    summary.to_excel(xw, sheet_name='Best settings', index=False); conv.to_excel(xw, sheet_name='Convergence', index=False)
print('✅ Saved optimal_hyperparameters_v2.json and Tuning_v2.xlsx')
